# Phase 2 - Généralisation en une famille de problèmes

## Définition mathématique

Un problème de la famille **"Achat de lots fractionnables"** se définit par :

- Une matrice $A \in \mathbb{R}^{m \times n}$ : composition des lots (lignes = armes, colonnes = lots)
- Un vecteur $b \in \mathbb{R}^m_+$ : demandes minimales en chaque arme
- Un vecteur $c \in \mathbb{R}^n_+$ : coûts des lots

Le problème primal général s'écrit :

$$
\min_{x \geq 0} \quad c^\top x \quad \text{sous} \quad A x \geq b
$$

**Avantage** : une seule classe Python pourra résoudre n'importe quelle instance de cette famille.

In [1]:
import pulp
import numpy as np
from typing import List, Dict, Any, Optional

class LotProblem:
    """
    Classe générale pour modéliser et résoudre un problème d'achat de lots fractionnables.
    
    Cette classe permet de résoudre n'importe quel problème de la forme :
    min c'x  s.t.  A x >= b,  x >= 0
    """
    
    def __init__(self, A: List[List[float]], b: List[float], c: List[float], name: str = "LotProblem"):
        """
        Initialise le problème.
        
        Parameters
        ----------
        A : list of lists
            Matrice de composition des lots (lignes = armes, colonnes = lots)
        b : list
            Demandes minimales pour chaque arme
        c : list
            Coûts des lots
        name : str
            Nom du problème (pour l'affichage)
        """
        self.A = np.array(A, dtype=float)
        self.b = np.array(b, dtype=float)
        self.c = np.array(c, dtype=float)
        self.name = name
        
        self._solution: Optional[List[float]] = None
        self._objective_value: Optional[float] = None
        
        # Vérification de cohérence des dimensions
        if self.A.shape[0] != len(self.b):
            raise ValueError(f"Nombre de lignes de A ({self.A.shape[0]}) doit correspondre à la taille de b ({len(self.b)})")
        if self.A.shape[1] != len(self.c):
            raise ValueError(f"Nombre de colonnes de A ({self.A.shape[1]}) doit correspondre à la taille de c ({len(self.c)})")
    
    def solve(self) -> Dict[str, Any]:
        """
        Résout le problème linéaire primal.
        
        Returns
        -------
        dict contenant le statut, la valeur objective et la solution
        """
        prob = pulp.LpProblem(f"{self.name}_Primal", pulp.LpMinimize)
        
        # Création des variables de décision
        n_lots = self.A.shape[1]
        x = [pulp.LpVariable(f"lot_{i+1}", lowBound=0) for i in range(n_lots)]
        
        # Fonction objectif
        prob += pulp.lpSum(self.c[i] * x[i] for i in range(n_lots))
        
        # Contraintes de demande
        for i in range(self.A.shape[0]):
            prob += pulp.lpSum(self.A[i, j] * x[j] for j in range(n_lots)) >= self.b[i], f"Demande_{i+1}"
        
        # Résolution
        status = prob.solve(pulp.PULP_CBC_CMD(msg=False))
        
        # Stockage des résultats
        self._solution = [x[i].varValue for i in range(n_lots)]
        self._objective_value = pulp.value(prob.objective)
        
        return {
            "status": pulp.LpStatus[status],
            "objective_value": round(self._objective_value, 4) if self._objective_value else None,
            "solution": [round(val, 4) for val in self._solution] if self._solution else None,
            "x": self._solution
        }
    
    def get_solution_summary(self) -> str:
        """Retourne un résumé lisible de la solution."""
        if self._solution is None:
            self.solve()
        
        summary = f"\n=== Solution du problème {self.name} ===\n"
        summary += f"Coût total       : {self._objective_value:.2f} M$\n"
        summary += "Quantités de lots :\n"
        for i, val in enumerate(self._solution, 1):
            summary += f"   Lot {i}     : {val:.4f}\n"
        return summary

In [2]:
# ====================== TEST AVEC LE PROBLÈME PATIBULAIRE ======================

A = [
    [500, 300, 800],    # Fusils
    [1000, 2000, 1500], # Grenades
    [10, 20, 15],       # Chars
    [100, 80, 15],      # Mitrailleuses
    [80, 120, 200]      # Bazookas
]

b = [100000, 200000, 100, 400, 400]
c = [10, 12, 15]

# Création de l'instance
problem = LotProblem(A, b, c, name="Patibulaire")

# Résolution
result = problem.solve()

print(problem.get_solution_summary())


=== Solution du problème Patibulaire ===
Coût total       : 1930.43 M$
Quantités de lots :
   Lot 1     : 0.0000
   Lot 2     : 8.6957
   Lot 3     : 121.7391



In [3]:
# ====================== TEST DE GÉNÉRALISATION - Autre exemple ======================

print("=== Test de généralisation sur un autre problème ===\n")

# Exemple plus simple : 2 armes, 2 lots
A2 = [
    [100, 200],   # Arme 1
    [150, 50]     # Arme 2
]

b2 = [5000, 3000]   # Demandes
c2 = [25, 18]       # Coûts des lots

problem2 = LotProblem(A2, b2, c2, name="Test_Généralisation")

result2 = problem2.solve()
print(problem2.get_solution_summary())

# Vérification manuelle rapide
print(f"Coût calculé manuellement pour vérification :")
print(f"Lot 1 : {result2['solution'][0]:.2f} × 25 = {result2['solution'][0]*25:.2f}")
print(f"Lot 2 : {result2['solution'][1]:.2f} × 18 = {result2['solution'][1]*18:.2f}")

=== Test de généralisation sur un autre problème ===


=== Solution du problème Test_Généralisation ===
Coût total       : 674.00 M$
Quantités de lots :
   Lot 1     : 14.0000
   Lot 2     : 18.0000

Coût calculé manuellement pour vérification :
Lot 1 : 14.00 × 25 = 350.00
Lot 2 : 18.00 × 18 = 324.00
